# 8. Evaluation

Per outlet, from the saved article outputs. The two approaches in the same two spaces:

| | levels | score |
|---|---|---|
| classifier | majority level | expected score (probabilities × mean training score of each level) |
| regressor | NewsGuard bounds · training quotas · article vote | mean score · score remapped to training quantiles |
| no text | most frequent level | mean training score |

95% intervals: bootstrap over outlets (fixed seed). Classifier vs regressor: paired, on the same outlets.

In [1]:
MODELS = "TF-IDF,Qwen3-8B,Mistral-24B,ModernBERT"
B = 2000

In [2]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import metrics as ME
import models as MO

art, outlets = MO.load()

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [3]:
VIEWS = {}                                         # (model, approach, view) -> (space, true, predicted)
for name in MODELS.split(","):
    pa = MO.predictions("classification", name)
    if pa is not None:
        T = ME.outlets_classifier(pa, outlets)
        VIEWS.update({(name, "classifier", v): x for v, x in ME.classifier_views(T, outlets).items()})
    pa = MO.predictions("regression", name)
    if pa is not None:
        T = ME.outlets_regressor(pa, outlets)
        VIEWS.update({(name, "regressor", v): x for v, x in ME.regressor_views(T, pa, art).items()})
VIEWS.update({("no text", "baseline", v): x for v, x in ME.baseline_views(outlets).items()})
rows = [{"model": m, "approach": a, "view": v, "space": s, **ME.METRICS[s](y, p)} for (m, a, v), (s, y, p) in VIEWS.items()]
all_metrics = pd.DataFrame(rows); all_metrics.to_csv(C.OUT / "all_metrics.csv", index=False)
print(f"{len(all_metrics)} views; models: {sorted({m for m, _, _ in VIEWS})}")

30 views; models: ['Mistral-24B', 'ModernBERT', 'Qwen3-8B', 'TF-IDF', 'no text']


In [4]:
all_metrics[all_metrics.space == "levels"].dropna(axis=1, how="all").set_index(["model", "approach", "view"]).round(3)

space  F1 macro  accuracy  balanced accuracy  tolerant (±1 level)  mean level error  kappa  quadratic kappa  Spearman  Kendall  F1 weighted  F1 l1  \
model       approach   view                                                                                                                                                                            
TF-IDF      classifier majority level            levels     0.320     0.403              0.328                0.796             0.863  0.179            0.558     0.556    0.476        0.362  0.451   
            regressor  levels: NewsGuard bounds  levels     0.239     0.314              0.316                0.836             0.850  0.148            0.496     0.619    0.545        0.277  0.056   
                       levels: training quotas   levels     0.385     0.425              0.386                0.836             0.761  0.254            0.679     0.682    0.573        0.424  0.423   
                       levels: article vote      levels     0.307     0.389              0.327                0.810             0.858  0.174            0.590     0.580    0.491        0.353  0.472   
Qwen3-8B    classifier majority level            levels     0.404     0.434              0.409                0.823             0.788  0.266            0.658     0.674    0.570        0.431  0.452   
            regressor  levels: NewsGuard bounds  levels     0.275     0.385              0.347                0.894             0.721  0.202            0.634     0.712    0.627        0.328  0.054   
                       levels: training quotas   levels     0.401     0.420              0.402                0.894             0.708  0.248            0.725     0.716    0.608        0.420  0.507   
                       levels: article vote      levels     0.316     0.394              0.334                0.841             0.805  0.187            0.647     0.632    0.535        0.364  0.358   
Mistral-24B classifier majority level            levels     0.388     0.429              0.389                0.827             0.796  0.247            0.625     0.626    0.539        0.420  0.494   
            regressor  levels: NewsGuard bounds  levels     0.276     0.372              0.342                0.876             0.752  0.185            0.611     0.683    0.600        0.319  0.105   
                       levels: training quotas   levels     0.420     0.438              0.421                0.858             0.726  0.271            0.700     0.699    0.594        0.437  0.479   
                       levels: article vote      levels     0.379     0.438              0.392                0.850             0.743  0.254            0.685     0.683    0.583        0.416  0.457   
ModernBERT  classifier majority level            levels     0.256     0.305              0.257                0.739             1.075  0.074            0.378     0.373    0.308        0.291  0.273   
            regressor  levels: NewsGuard bounds  levels     0.255     0.354              0.322                0.814             0.845  0.166            0.488     0.555    0.480        0.305  0.051   
                       levels: training quotas   levels     0.341     0.358              0.343                0.810             0.876  0.167            0.609     0.604    0.505        0.358  0.507   
                       levels: article vote      levels     0.315     0.363              0.324                0.765             0.947  0.152            0.537     0.545    0.463        0.345  0.459   
no text     baseline   most frequent level       levels     0.104     0.350              0.200                0.655             1.150  0.000            0.000       NaN      NaN        0.181  0.000   

                                                 F1 l2  F1 l3  F1 l4  F1 l5  reliable: accuracy  reliable: F1 macro  reliable: MCC  
model       approach   view                                                                                         

In [5]:
all_metrics[all_metrics.space == "score"].dropna(axis=1, how="all").set_index(["model", "approach", "view"]).round(3)

space  Spearman  Kendall  reliable: accuracy  reliable: F1 macro     MAE    RMSE  median error     R²  Pearson  Lin's CCC  bias (predicted − true)  \
model       approach   view                                                                                                                                                                      
TF-IDF      classifier expected score       score     0.682    0.494               0.765               0.725  16.582  21.007        15.138  0.397    0.655      0.508                    0.244   
            regressor  mean score           score     0.691    0.496               0.788               0.745  17.651  21.834        15.775  0.348    0.660      0.427                   -1.205   
                       remapped score       score     0.696    0.518               0.783               0.762  16.287  22.021        12.500  0.337    0.677      0.676                   -0.400   
Qwen3-8B    classifier expected score       score     0.715    0.520               0.832               0.816  15.930  20.023        13.407  0.452    0.703      0.565                   -1.497   
            regressor  mean score           score     0.722    0.526               0.832               0.814  16.072  20.085        13.070  0.449    0.699      0.560                   -0.740   
                       remapped score       score     0.726    0.544               0.827               0.811  15.293  21.230        12.500  0.384    0.700      0.699                   -0.400   
Mistral-24B classifier expected score       score     0.727    0.538               0.796               0.776  14.769  18.918        12.328  0.511    0.723      0.646                   -0.486   
            regressor  mean score           score     0.735    0.538               0.814               0.793  15.736  19.577        13.374  0.476    0.718      0.589                   -0.861   
                       remapped score       score     0.735    0.554               0.819               0.801  15.248  20.806        12.500  0.408    0.711      0.711                   -0.400   
ModernBERT  classifier expected score       score     0.576    0.403               0.757               0.714  18.300  22.819        16.872  0.288    0.548      0.412                   -1.519   
            regressor  mean score           score     0.605    0.430               0.765               0.725  17.677  22.027        14.795  0.337    0.588      0.466                    0.054   
                       remapped score       score     0.602    0.444               0.748               0.723  18.131  24.022        13.000  0.211    0.615      0.615                   -0.400   
no text     baseline   mean training score  score    -0.015   -0.012               0.650               0.394  22.575  27.049        19.175 -0.000   -0.014     -0.000                   -0.001   

                                            within 10 points  within 20 points  reliable: AUC  
model       approach   view                                                                    
TF-IDF      classifier expected score                  0.372             0.677          0.850  
            regressor  mean score                      0.323             0.611          0.865  
                       remapped score                  0.412             0.708          0.867  
Qwen3-8B    classifier expected score                  0.381             0.686          0.876  
            regressor  mean score                      0.354             0.690          0.883  
                       remapped score                  0.447             0.757          0.879  
Mistral-24B classifier expected score                  0.429             0.748          0.863  
            regressor  mean score                      0.367             0.686          0.873  
                       remapped score                  0.442             0.752          0.871  
ModernBERT  classifier expected score                  0.305             0.60

## Intervals

In [6]:
KEYS = {"levels": ["F1 macro", "tolerant (±1 level)", "quadratic kappa", "Spearman", "reliable: accuracy"],
        "score": ["MAE", "Spearman", "within 10 points", "within 20 points", "reliable: accuracy"]}
rows = []
for (m, a, v), (s, y, p) in VIEWS.items():
    ci, value = ME.bootstrap(y, p, s, KEYS[s], B=B), ME.METRICS[s](y, p)
    rows += [{"model": m, "approach": a, "view": v, "space": s, "metric": c, "value": value[c], "low": ci[c][0],
              "high": ci[c][1]} for c in KEYS[s]]
intervals = pd.DataFrame(rows); intervals.to_csv(C.OUT / "intervals.csv", index=False)
print(len(intervals), "intervals")

150 intervals


## Classifier vs regressor (regressor − classifier, same outlets)

In [7]:
PAIRS = {"levels": ("majority level", ["levels: NewsGuard bounds", "levels: training quotas", "levels: article vote"]),
         "score": ("expected score", ["mean score", "remapped score"])}
rows = []
for name in MODELS.split(","):
    for space, (cls_view, reg_views) in PAIRS.items():
        if (name, "classifier", cls_view) not in VIEWS or (name, "regressor", reg_views[0]) not in VIEWS:
            continue
        for reg_view in reg_views:
            for key in KEYS[space]:
                d, (lo, hi) = ME.bootstrap_difference(VIEWS[(name, "regressor", reg_view)], VIEWS[(name, "classifier", cls_view)], key, B=B)
                rows.append({"model": name, "space": space, "regressor view": reg_view, "metric": key,
                             "difference": d, "low": lo, "high": hi, "not chance": not (lo <= 0 <= hi)})
pairs = pd.DataFrame(rows); pairs.to_csv(C.OUT / "classifier_vs_regressor.csv", index=False)
pairs.round(3)

,model,space,regressor view,metric,difference,low,high,not chance
0,TF-IDF,levels,levels: NewsGuard bounds,F1 macro,-0.081,-0.154,-0.003,True
1,TF-IDF,levels,levels: NewsGuard bounds,tolerant (±1 level),0.040,-0.027,0.106,False
2,TF-IDF,levels,levels: NewsGuard bounds,quadratic kappa,-0.062,-0.150,0.030,False
3,TF-IDF,levels,levels: NewsGuard bounds,Spearman,0.063,-0.018,0.150,False
4,TF-IDF,levels,levels: NewsGuard bounds,reliable: accuracy,0.044,-0.004,0.093,False
...,...,...,...,...,...,...,...,...
95,ModernBERT,score,remapped score,MAE,-0.169,-2.143,1.799,False
96,ModernBERT,score,remapped score,Spearman,0.026,-0.030,0.079,False
97,ModernBERT,score,remapped score,within 10 points,0.062,-0.027,0.150,False
98,ModernBERT,score,remapped score,within 20 points,0.062,-0.013,0.137,False


## Probability metrics of the classifiers

In [8]:
pd.DataFrame({name: ME.probability_metrics(ME.outlets_classifier(MO.predictions("classification", name), outlets))
              for name in MODELS.split(",") if MO.predictions("classification", name) is not None}).T.round(3)

,AUC macro (5 levels),log loss,Brier,reliable: AUC (probabilities)
TF-IDF,0.684,1.460,0.737,0.834
Qwen3-8B,0.727,1.362,0.700,0.846
Mistral-24B,0.714,1.521,0.711,0.844
ModernBERT,0.633,1.582,0.807,0.785
